In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import os
import shutil
from tqdm import tqdm
from IPython.display import FileLink

# 1. CLEANING THE WORKING DIRECTORY
def clear_working_dir(path='/kaggle/working'):
    print(f"Cleaning {path}...")
    for item in os.listdir(path):
        item_path = os.path.join(path, item)
        try:
            if os.path.isfile(item_path) or os.path.islink(item_path):
                os.unlink(item_path) # Delete files or links
            elif os.path.isdir(item_path):
                shutil.rmtree(item_path) # Delete folders
        except Exception as e:
            print(f'Failed to delete {item_path}. Reason: {e}')
    print("Working directory cleared.")

# 2. RECURSIVE MERGE LOGIC
def merge_datasets_recursive(source_dir, prefix, output_folder):
    print(f"\nSearching for images in: {source_dir}")
    
    # Locate all image files in any subfolder
    valid_extensions = ('.png', '.jpg', '.jpeg', '.webp', '.bmp', '.tiff')
    all_files = []
    for root, dirs, files in os.walk(source_dir):
        for file in files:
            if file.lower().endswith(valid_extensions):
                all_files.append(os.path.join(root, file))
    
    if not all_files:
        print(f"Warning: No images found in {source_dir}. Check the path!")
        return

    print(f"Found {len(all_files)} images. Copying...")
    for src_file in tqdm(all_files):
        filename = os.path.basename(src_file)
        # Prefix prevents overwriting if different datasets use same names (e.g., 01.jpg)
        new_filename = f"{prefix}_{filename}"
        dst_file = os.path.join(output_folder, new_filename)
        shutil.copy2(src_file, dst_file)

# --- EXECUTION ---

# Paths provided by you
ds1_path = '/kaggle/input/datasets/debanikkk/final-1mp-omni-hq50-6928'
ds2_path = '/kaggle/input/datasets/somanshu7527/style-trans-dl-2913images'
output_folder = '/kaggle/working/combined_images'

# Step 1: Wipe everything currently in /kaggle/working
clear_working_dir()

# Step 2: Create the fresh output folder
os.makedirs(output_folder, exist_ok=True)

# Step 3: Run the recursive merge
merge_datasets_recursive(ds1_path, "ds1", output_folder)
merge_datasets_recursive(ds2_path, "ds2", output_folder)

# Step 4: Verification and Zipping
total_images = len(os.listdir(output_folder))
print(f"\n--- Final Results ---")
print(f"Total images in combined folder: {total_images}")

if total_images > 0:
    print("Zipping dataset... this may take a minute.")
    shutil.make_archive('combined_dataset', 'zip', output_folder)
    print("Zip complete.")
    display(FileLink(r'combined_dataset.zip'))

Cleaning /kaggle/working...
Working directory cleared.

Searching for images in: /kaggle/input/datasets/debanikkk/final-1mp-omni-hq50-6928
Found 6928 images. Copying...


100%|██████████| 6928/6928 [00:30<00:00, 228.24it/s]



Searching for images in: /kaggle/input/datasets/somanshu7527/style-trans-dl-2913images
Found 2913 images. Copying...


100%|██████████| 2913/2913 [00:30<00:00, 95.75it/s] 



--- Final Results ---
Total images in combined folder: 9841
Zipping dataset... this may take a minute.
Zip complete.


/kaggle/working/combined_dataset.zip

In [1]:
import os
from PIL import Image
from tqdm import tqdm

# 1. Define the path to the specific dataset
dataset_path = '/kaggle/input/datasets/somanshu7527/style-trans-dl-2913images'
target_size = (1024, 1024)

# 2. Setup counters and lists
mismatched = []
total_images = 0
valid_extensions = ('.png', '.jpg', '.jpeg', '.webp', '.bmp')

print(f"Scanning all folders in: {dataset_path}...")

# 3. Recursive Search using os.walk
for root, dirs, files in os.walk(dataset_path):
    for file in files:
        if file.lower().endswith(valid_extensions):
            total_images += 1
            file_path = os.path.join(root, file)
            
            try:
                with Image.open(file_path) as img:
                    if img.size != target_size:
                        mismatched.append({
                            'name': file,
                            'size': img.size,
                            'path': file_path
                        })
            except Exception as e:
                print(f"Error opening {file}: {e}")

# 4. Final Report
print(f"\n--- Scan Complete ---")
print(f"Total images found and checked: {total_images}")

if total_images == 0:
    print("Warning: No images were found. Please double-check the dataset path.")
elif not mismatched:
    print(f"Perfect! All {total_images} images are exactly {target_size[0]}x{target_size[1]}.")
else:
    print(f"Found {len(mismatched)} images with incorrect dimensions:")
    # Display the first 10 mismatches
    for i, item in enumerate(mismatched[:10]):
        print(f"[{i+1}] {item['name']} - Size: {item['size']}")
    
    if len(mismatched) > 10:
        print(f"... and {len(mismatched) - 10} more.")

Scanning all folders in: /kaggle/input/datasets/somanshu7527/style-trans-dl-2913images...

--- Scan Complete ---
Total images found and checked: 2913
Found 536 images with incorrect dimensions:
[1] 540_00029_c7997e897e.jpg - Size: (768, 1024)
[2] 513_00013_a2f82a4752.jpg - Size: (1024, 683)
[3] 5_00002_c5933922b3.jpg - Size: (1024, 768)
[4] 12_00015_7039663f10.jpg - Size: (768, 1024)
[5] 277_00015_33b5ca9724.jpg - Size: (768, 1024)
[6] 109_00021_a5d1637669.jpg - Size: (1024, 768)
[7] 240_00006_336436b326.jpg - Size: (1024, 768)
[8] 383_00021_a8757780aa.jpg - Size: (1024, 769)
[9] 322_00002_aef90c0ac3.jpg - Size: (927, 716)
[10] 327_00003_28e07e4971.jpg - Size: (1024, 427)
... and 526 more.


In [2]:
import os
from PIL import Image
from tqdm import tqdm

# Configuration
input_path = '/kaggle/input/datasets/somanshu7527/style-trans-dl-2913images/kaggle/working/combined_dataset'
output_dir = '/kaggle/working/processed_images'
os.makedirs(output_dir, exist_ok=True)

target_size = (1024, 1024)
min_dim = 1000
valid_extensions = ('.png', '.jpg', '.jpeg', '.webp', '.bmp')

processed_count = 0
square_large_count = 0

print(f"Scanning images in: {input_path}...")

# Get list of all files first for tqdm progress bar
all_files = []
for root, _, files in os.walk(input_path):
    for file in files:
        if file.lower().endswith(valid_extensions):
            all_files.append(os.path.join(root, file))

for file_path in tqdm(all_files, desc="Processing Images"):
    try:
        with Image.open(file_path) as img:
            width, height = img.size
            
            # Condition: Square AND dimensions > 1000x1000
            if width == height and width > min_dim:
                square_large_count += 1
                save_path = os.path.join(output_dir, os.path.basename(file_path))
                
                # Check if already 1024x1024
                if img.size == target_size:
                    img.save(save_path)
                else:
                    # Resize using Lanczos resampling
                    resized_img = img.resize(target_size, resample=Image.Resampling.LANCZOS)
                    resized_img.save(save_path)
                
                processed_count += 1
    except Exception as e:
        print(f"Error processing {file_path}: {e}")

print(f"\n--- Process Complete ---")
print(f"Square images > 1000px found: {square_large_count}")
print(f"Images saved to {output_dir}: {processed_count}")

Scanning images in: /kaggle/input/datasets/somanshu7527/style-trans-dl-2913images/kaggle/working/combined_dataset...


Processing Images: 100%|██████████| 2913/2913 [00:35<00:00, 81.53it/s]


--- Process Complete ---
Square images > 1000px found: 2377
Images saved to /kaggle/working/processed_images: 2377


In [3]:
import shutil

# The folder where your images are stored
source_dir = '/kaggle/working/processed_images'
# The name of the zip file to create (will result in processed_images_archive.zip)
output_filename = '/kaggle/working/processed_images_archive'

shutil.make_archive(output_filename, 'zip', source_dir)

print(f"Zip file created at: {output_filename}.zip")

Zip file created at: /kaggle/working/processed_images_archive.zip


In [4]:
from IPython.display import FileLink

# Generate a link to download the zip file
FileLink(r'processed_images_archive.zip')

/kaggle/working/processed_images_archive.zip